In [108]:
%load_ext autoreload
%autoreload 2


from libthesis import pdf_writer, update_layout

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [109]:
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import pandas

def calc_graph(k):
    # Load and filter
    df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")
    kmer = k
    df = df[df["kmer_size"] == kmer].copy()
    df = df[df["cogent3_pd"].notnull() & df["madb_cycles"].notnull()].copy()

    # Ensure only matched stable_ids across Chimpanzee and Macaque
    chimp_ids = set(df[df["species"] == "Chimpanzee"]["unique_id"])
    macaque_ids = set(df[df["species"] == "Macaque"]["unique_id"])
    matched_ids = chimp_ids & macaque_ids

    df = df[df["unique_id"].isin(matched_ids) & df["species"].isin(["Chimpanzee", "Macaque"])]

    # Calculate speed as kbp per second
    df["madb_kbp_per_sec"] = (df["seq_length"] / 1000) / df["madb_time"]

    # Plot config
    species_colors = {
        "Chimpanzee": "#CC0000",  # Red
        "Macaque":    "#0072B2",  # Blue
    }

    fig = go.Figure()
    summary_stats = {}

    for species in ["Chimpanzee", "Macaque"]:
        species_df = df[df["species"] == species]
        x = species_df["cogent3_pd"].values.reshape(-1, 1)
        y = species_df["madb_kbp_per_sec"].values

        # Linear regression
        model = LinearRegression().fit(x, y)
        y_pred = model.predict(x)
        slope = model.coef_[0]
        r2 = r2_score(y, y_pred)

        summary_stats[species] = (slope, r2)

        fig.add_trace(go.Scatter(
            x=species_df["cogent3_pd"],
            y=species_df["madb_kbp_per_sec"],
            mode="markers",
            marker=dict(color=species_colors[species]),
            name=f"Human–{species}"
        ))

        x_range = pandas.Series(sorted(species_df["cogent3_pd"].unique()))
        fig.add_trace(go.Scatter(
            x=x_range,
            y=model.predict(x_range.values.reshape(-1, 1)),
            mode="lines",
            line=dict(dash="dot", color=species_colors[species]),
            showlegend=False
        ))

    # Annotation box
    annotation_text = "<br>".join(
        f"Human–{species}: slope = {slope:.3f}, R² = {r2:.3f}"
        for species, (slope, r2) in summary_stats.items()
    )
    fig.add_annotation(
        text=annotation_text,
        xref="paper", yref="paper",
        x=0.98, y=0.98,
        showarrow=False,
        xanchor="right", yanchor="top",
        font=dict(size=14),
        align="right",
        bordercolor="black",
        borderwidth=1,
        borderpad=4,
        bgcolor="white",
        opacity=0.9
    )

    fig.update_layout(
        xaxis_title="Genetic distance (PD)",
        yaxis_title="Speed (kbp/s)",
        width=800,
        height=600,
        legend=dict(x=0.01, y=0.99),
        margin=dict(l=40, r=20, t=40, b=40),
    )

    update_layout(fig, in_panel=False)
    fig.show()

    write_pdf = pdf_writer()
    write_pdf(fig, "madb_speed_vs_divergence_matched")

calc_graph(15)

In [110]:
import pandas
from scipy.stats import mannwhitneyu
from pathlib import Path

def calc_stats(k : int = 15):
    # Load and filter dataset
    df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")
    kmer = k
    df = df[df["kmer_size"] == kmer].copy()

    # Compute alignment speed in kilobases per second
    df["madb_kbp_per_sec"] = (df["seq_length"] / 1000) / df["madb_time"]

    # Keep rows with matched stable IDs in both Chimpanzee and Macaque
    chimp_ids = set(df[df["species"] == "Chimpanzee"]["unique_id"])
    macaque_ids = set(df[df["species"] == "Macaque"]["unique_id"])
    common_ids = chimp_ids & macaque_ids
    df = df[df["unique_id"].isin(common_ids)].copy()

    # Prepare rows for LaTeX table
    rows = []
    for species in ["Chimpanzee", "Macaque"]:
        sub = df[df["species"] == species]
        q1 = sub["cogent3_pd"].quantile(0.25)
        q4 = sub["cogent3_pd"].quantile(0.75)

        low = sub[sub["cogent3_pd"] <= q1]["madb_kbp_per_sec"]
        high = sub[sub["cogent3_pd"] >= q4]["madb_kbp_per_sec"]

        u_stat, u_pval = mannwhitneyu(low, high, alternative="two-sided")

        stars = (
            "$^*$" if u_pval < 0.05 and u_pval >= 0.01 else
            "$^{**}$" if u_pval < 0.01 and u_pval >= 0.001 else
            "$^{***}$" if u_pval < 0.001 else ""
        )

        # First row: low group with p-value
        rows.append({
            "Primate species": species,
            "Group": "Low ($\\leq$ Q1)",
            "N": len(low),
            "Mean (kbp/s)": low.mean(),
            "Median (kbp/s)": low.median(),
            "Std Dev": low.std(),
            "p-value": f"\\multirow{{2}}{{*}}{{{u_pval:.3f}}}",
            " ": f"\\multirow{{2}}{{*}}{{{stars}}}"
        })

        # Second row: high group with empty p-value/star
        rows.append({
            "Primate species": species,
            "Group": "High ($\\geq$ Q4)",
            "N": len(high),
            "Mean (kbp/s)": high.mean(),
            "Median (kbp/s)": high.median(),
            "Std Dev": high.std(),
            "p-value": "",
            " ": ""
        })
        print(f"{species} {u_pval:.3f}")

    # Generate LaTeX table
    latex_table = pandas.DataFrame(rows).to_latex(
        index=False,
        column_format="llrrrrr@{\\,}l",
        float_format="%.3f",
        escape=False,
        multicolumn=False
    )

    # Write to figures directory
    figures_path = Path("..") / "figures"
    figures_path.mkdir(parents=True, exist_ok=True)
    with open(figures_path / "speed_vs_divergence_table.tex", "w") as f:
        f.write(latex_table)

calc_stats(15)

Chimpanzee 0.100
Macaque 0.400


In [111]:
import pandas
from scipy.stats import wilcoxon
from pathlib import Path

def calc_paired_stats(k):

    # Load and filter dataset
    df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")
    kmer = 95
    df = df[df["kmer_size"] == kmer].copy()
    df = df[df["madb_time"].notnull()]

    # Split into matched species and align on stable_id
    chimp_df = df[df["species"] == "Chimpanzee"].set_index("unique_id")
    macaque_df = df[df["species"] == "Macaque"].set_index("unique_id")
    common_ids = chimp_df.index.intersection(macaque_df.index)

    chimp_time = chimp_df.loc[common_ids, "madb_time"]
    macaque_time = macaque_df.loc[common_ids, "madb_time"]

    # Compute paired differences
    diff = macaque_time - chimp_time
    mean_diff = diff.mean()
    n = len(diff)

    # Perform Wilcoxon signed-rank test
    stat, p_value = wilcoxon(diff)

    # Format significance
    stars = (
        "$^*$" if p_value < 0.05 and p_value >= 0.01 else
        "$^{**}$" if p_value < 0.01 and p_value >= 0.001 else
        "$^{***}$" if p_value < 0.001 else ""
    )

    # Compose LaTeX table
    latex_table = rf"""
    \begin{{tabular}}{{lrrrc}}
    \toprule
    \textbf{{Comparison}} & \textbf{{$N$ pairs}} & \textbf{{Mean Diff (s)}} & \textbf{{Test}} & \textbf{{$p$-value}} \\
    \midrule
    Human–Macaque $-$ Human–Chimpanzee & {n} & {mean_diff:.3f} & Wilcoxon & ${p_value:.1e}{stars}$ \\
    \bottomrule
    \end{{tabular}}
    """.strip()

    # Save to .tex file
    figures_path = Path("..") / "figures" 
    figures_path.mkdir(parents=True, exist_ok=True)
    output_file = figures_path / "paired_speed_test_table.tex"

    with open(output_file, "w") as f:
        f.write(latex_table)

    print(f"Number of pairs: {n}")
    print(f"Mean difference: {mean_diff:.4f} s")
    print(f"Wilcoxon signed-rank test p-value: {p_value:.1e} {stars}")

calc_paired_stats(95)

Number of pairs: 22
Mean difference: 1.1144 s
Wilcoxon signed-rank test p-value: 9.1e-06 $^{***}$


In [112]:
k = 95
calc_paired_stats(k)
calc_stats(k)
calc_graph(k)


Number of pairs: 22
Mean difference: 1.1144 s
Wilcoxon signed-rank test p-value: 9.1e-06 $^{***}$
Chimpanzee 0.041
Macaque 0.009
